<div style="background-color: #0b1f1a; padding: 25px; border-left: 10px solid #00f59f; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
    <h1 style="color: #ffffff; margin: 0; font-family: 'Segoe UI', Arial, sans-serif; font-size: 32px;">
        💰 Revenue Analysis Page Validation
    </h1>
    <p style="color: #b0c9c2; margin: 10px 0 0 0; font-size: 16px; font-family: 'Segoe UI', Arial, sans-serif;">
        Automated SQL Data Extraction & Revenue Performance Verification Pipeline
    </p>
</div>

In [1]:
import os
import boto3
import duckdb
import pandas as pd
import dataframe_image as dfi
from io import BytesIO
from dotenv import load_dotenv

# 1. Load AWS Credentials from .env
load_dotenv(dotenv_path="../../.env")
AWS_REGION = os.getenv("AWS_REGION")
PROCESSED_BUCKET = 'ra2-processed-data'

# 2. Setup revenue_analysis_output folder
revenue_output_folder = "../../eda_outputs/revenue_analysis_output"
os.makedirs(revenue_output_folder, exist_ok=True)

# 3. Connect to S3 securely
s3 = boto3.client('s3', region_name=AWS_REGION)
print("🔗 Connecting to S3 Processed Bucket for Revenue Analysis...\n")

# 4. Download ALL files from S3 and explicitly register them in DuckDB
response = s3.list_objects_v2(Bucket=PROCESSED_BUCKET)

print("📥 Downloading and Registering Clean Tables:")
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    if file_key.endswith('.csv'):
        table_name = file_key.replace('.csv', '').lower()
        print(f"   -> Fetching {file_key} and registering as table: '{table_name}'...")
        
        csv_obj = s3.get_object(Bucket=PROCESSED_BUCKET, Key=file_key)
        df_temp = pd.read_csv(BytesIO(csv_obj['Body'].read()))
        
        # Explicitly register the dataframe into DuckDB
        duckdb.register(table_name, df_temp)

print("\n✅ All clean data successfully registered in DuckDB for Revenue Analysis!")

# 5. DuckDB Validation Function (Saves to Revenue Analysis Output Folder)
def save_sql_to_image(sql_query, image_name):
    print(f"Executing query for: {image_name}...")
    df = duckdb.query(sql_query).df()
    
    file_path = os.path.join(revenue_output_folder, f"{image_name}.png")
    dfi.export(df, file_path, table_conversion="matplotlib")
    
    print(f"✅ Image saved successfully at: {file_path}\n")
    return df

c:\Users\pushp\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.6.0)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


🔗 Connecting to S3 Processed Bucket for Revenue Analysis...

📥 Downloading and Registering Clean Tables:
   -> Fetching customers.csv and registering as table: 'customers'...
   -> Fetching delivery.csv and registering as table: 'delivery'...
   -> Fetching feedback.csv and registering as table: 'feedback'...
   -> Fetching menu_items.csv and registering as table: 'menu_items'...
   -> Fetching order_items.csv and registering as table: 'order_items'...
   -> Fetching orders.csv and registering as table: 'orders'...
   -> Fetching outlets.csv and registering as table: 'outlets'...
   -> Fetching promotions.csv and registering as table: 'promotions'...

✅ All clean data successfully registered in DuckDB for Revenue Analysis!


In [2]:
import os
import dataframe_image as dfi

# --- 1. Total Revenue KPI ---
df_rev = duckdb.query("""
    SELECT ROUND(SUM(final_amount) / 1000000, 2) AS "Total Revenue (₹M)"
    FROM orders 
    WHERE order_status = 'Delivered';
""").df()
dfi.export(df_rev, os.path.join(revenue_output_folder, "1_kpi_total_revenue.png"), table_conversion="matplotlib")


# --- 2. Average Order Value (AOV) KPI ---
df_aov = duckdb.query("""
    SELECT ROUND(AVG(final_amount), 2) AS "Average Order Value (₹)"
    FROM orders 
    WHERE order_status = 'Delivered';
""").df()
dfi.export(df_aov, os.path.join(revenue_output_folder, "2_kpi_avg_order_value.png"), table_conversion="matplotlib")


# --- 3. Total Discounts Applied KPI ---
df_disc = duckdb.query("""
    SELECT ROUND(SUM(discount_applied), 2) AS "Total Discounts (₹)"
    FROM orders 
    WHERE order_status = 'Delivered';
""").df()
dfi.export(df_disc, os.path.join(revenue_output_folder, "3_kpi_total_discounts.png"), table_conversion="matplotlib")


# --- 4. Total Delivered Orders Count KPI ---
df_orders = duckdb.query("""
    SELECT COUNT(order_id) AS "Delivered Orders Count"
    FROM orders 
    WHERE order_status = 'Delivered';
""").df()
dfi.export(df_orders, os.path.join(revenue_output_folder, "4_kpi_delivered_orders.png"), table_conversion="matplotlib")


# --- 5. Peak Revenue City KPI ---
df_city = duckdb.query("""
    SELECT o.city AS "Top Revenue City", ROUND(SUM(ord.final_amount) / 1000000, 2) AS "Revenue (₹M)"
    FROM orders ord
    JOIN outlets o ON ord.outlet_id = o.outlet_id
    WHERE ord.order_status = 'Delivered'
    GROUP BY o.city
    ORDER BY "Revenue (₹M)" DESC
    LIMIT 1;
""").df()
dfi.export(df_city, os.path.join(revenue_output_folder, "5_kpi_top_revenue_city.png"), table_conversion="matplotlib")

print("✅ All 5 Revenue Analysis KPIs successfully generated and saved!")

✅ All 5 Revenue Analysis KPIs successfully generated and saved!


In [3]:
import matplotlib.pyplot as plt
import os

# 1. Query monthly revenue and order volume
df_combo = duckdb.query("""
    SELECT 
        strftime('%b', CAST(order_date AS DATE)) AS month_name,
        MONTH(CAST(order_date AS DATE)) AS month_num,
        ROUND(SUM(final_amount) / 1000000, 2) AS revenue_millions,
        COUNT(order_id) AS order_volume
    FROM orders
    WHERE order_status = 'Delivered' AND order_date IS NOT NULL
    GROUP BY month_name, month_num
    ORDER BY month_num ASC;
""").df()

# 2. Plot Dual-Axis Combo Chart
plt.figure(figsize=(12, 6), facecolor='#0b1f1a')
ax1 = plt.axes()
ax1.set_facecolor('#0b1f1a')

x_indices = range(len(df_combo))

# Left Y-axis: Revenue Bars (Neon Mint)
bars = ax1.bar(x_indices, df_combo['revenue_millions'], color='#00f59f', width=0.6, alpha=0.85)
ax1.set_ylabel('Revenue', color='#b0c9c2', fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', colors='#b0c9c2')
ax1.set_ylim(0, max(df_combo['revenue_millions']) * 1.25)

# Right Y-axis: Order Volume Line (White with markers)
ax2 = ax1.twinx()
ax2.set_facecolor('#0b1f1a')
line = ax2.plot(x_indices, df_combo['order_volume'], color='white', marker='o', linewidth=2.5, markersize=6)
ax2.set_ylabel('Order Volume', color='#b0c9c2', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', colors='#b0c9c2')

# X-Axis Styling
ax1.set_xticks(x_indices)
ax1.set_xticklabels(df_combo['month_name'], color='#b0c9c2', fontsize=10)

# Title and Grid
plt.title('Monthly revenue versus order volume', color='white', fontsize=15, fontweight='bold', pad=15)
ax1.grid(axis='y', linestyle='--', alpha=0.15, color='#b0c9c2')

# Border Spines Styling
for spine in ax1.spines.values():
    spine.set_color('#2e5e52')
for spine in ax2.spines.values():
    spine.set_color('#2e5e52')

plt.tight_layout()

# 3. Save Image
file_path = os.path.join(revenue_output_folder, "6_main_chart_monthly_revenue_vs_orders.png")
plt.savefig(file_path, facecolor='#0b1f1a', edgecolor='none', dpi=300)
plt.close()

print(f"✅ Main Revenue vs Order Volume chart successfully generated and saved at: {file_path}")

✅ Main Revenue vs Order Volume chart successfully generated and saved at: ../../eda_outputs/revenue_analysis_output\6_main_chart_monthly_revenue_vs_orders.png


In [4]:
import matplotlib.pyplot as plt
import os

# --- 1. Query Data for the 4 Charts ---

# Chart 1: Total Revenue by Order Type (Donut)
df_order_type = duckdb.query("""
    SELECT order_type, ROUND(SUM(final_amount) / 1000000, 2) AS revenue_m
    FROM orders WHERE order_status = 'Delivered'
    GROUP BY order_type ORDER BY revenue_m DESC;
""").df()

# Chart 2: Revenue by Week Type (Donut)
df_week_type = duckdb.query("""
    SELECT week_type, ROUND(SUM(final_amount) / 1000000, 2) AS revenue_m
    FROM orders WHERE order_status = 'Delivered' AND week_type IS NOT NULL
    GROUP BY week_type ORDER BY revenue_m DESC;
""").df()

# Chart 3: Sales by Fulfillment Status (Vertical Bar)
df_fulfillment = duckdb.query("""
    SELECT order_status, ROUND(SUM(final_amount) / 1000000, 2) AS revenue_m
    FROM orders
    GROUP BY order_status ORDER BY revenue_m DESC;
""").df()

# Chart 4: Transaction Volume by Payment Type (Horizontal Bar)
df_payment = duckdb.query("""
    SELECT payment_method, ROUND(SUM(final_amount) / 1000000, 2) AS revenue_m
    FROM orders WHERE order_status = 'Delivered'
    GROUP BY payment_method ORDER BY revenue_m ASC;
""").df()


# --- 2. Setup 2x2 Subplot Grid Matching Dashboard Theme ---
fig, axs = plt.subplots(2, 2, figsize=(14, 10), facecolor='#0b1f1a')
axs = axs.ravel()

for ax in axs:
    ax.set_facecolor('#0b1f1a')
    for spine in ax.spines.values():
        spine.set_color('#2e5e52')

# --- Plot 1: Total Revenue by Order Type (Donut) ---
wedges1, texts1, autotexts1 = axs[0].pie(
    df_order_type['revenue_m'], 
    labels=df_order_type['order_type'], 
    colors=['#00f59f', '#1dd1a1', '#10ac84'], 
    autopct='%1.1f%%', 
    startangle=90,
    textprops=dict(color='#b0c9c2', fontsize=9)
)
# Make it a Donut Chart
centre_circle1 = plt.Circle((0,0), 0.70, fc='#0b1f1a')
axs[0].add_artist(centre_circle1)
axs[0].set_title('Total Revenue by Order Type', color='white', fontsize=12, fontweight='bold', pad=10)


# --- Plot 2: Revenue by Week Type (Donut) ---
wedges2, texts2, autotexts2 = axs[1].pie(
    df_week_type['revenue_m'], 
    labels=df_week_type['week_type'], 
    colors=['#00f59f', '#81ecec'], 
    autopct='%1.1f%%', 
    startangle=90,
    textprops=dict(color='#b0c9c2', fontsize=9)
)
centre_circle2 = plt.Circle((0,0), 0.70, fc='#0b1f1a')
axs[1].add_artist(centre_circle2)
axs[1].set_title('Revenue by Week Type', color='white', fontsize=12, fontweight='bold', pad=10)


# --- Plot 3: Sales by Fulfillment Status (Vertical Bar) ---
axs[2].bar(df_fulfillment['order_status'], df_fulfillment['revenue_m'], color='#00f59f', width=0.55)
axs[2].set_title('Sales by Fulfillment Status', color='white', fontsize=12, fontweight='bold', pad=10)
axs[2].set_ylabel('Revenue (₹M)', color='#b0c9c2', fontsize=9)
axs[2].tick_params(axis='x', colors='#b0c9c2', rotation=15)
axs[2].tick_params(axis='y', colors='#b0c9c2')
axs[2].grid(axis='y', linestyle='--', alpha=0.15, color='#b0c9c2')


# --- Plot 4: Transaction Volume by Payment Type (Horizontal Bar) ---
axs[3].barh(df_payment['payment_method'], df_payment['revenue_m'], color='#00f59f', height=0.55)
axs[3].set_title('Transaction volume by payment type', color='white', fontsize=12, fontweight='bold', pad=10)
axs[3].set_xlabel('Revenue (₹M)', color='#b0c9c2', fontsize=9)
axs[3].tick_params(axis='x', colors='#b0c9c2')
axs[3].tick_params(axis='y', colors='#b0c9c2', labelsize=9)
axs[3].grid(axis='x', linestyle='--', alpha=0.15, color='#b0c9c2')


plt.tight_layout()

# --- 3. Save Combined Image ---
file_path = os.path.join(revenue_output_folder, "7_to_10_supportive_charts_revenue.png")
plt.savefig(file_path, facecolor='#0b1f1a', edgecolor='none', dpi=300)
plt.close()

print(f"✅ All 4 supportive revenue charts successfully generated and saved at: {file_path}")

✅ All 4 supportive revenue charts successfully generated and saved at: ../../eda_outputs/revenue_analysis_output\7_to_10_supportive_charts_revenue.png
